# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Leer credenciales desde Colab Secrets — soporte de implementación, fase 1.
import sys
import os
from dataclasses import dataclass, field
from pathlib import Path

LOCAL_TEST_MODE = "LOCAL_TEST_MODE"
WHATSAPP_MODE = "WHATSAPP_MODE"
MODE = LOCAL_TEST_MODE  # No consulta Secrets de Meta ni ngrok.
IN_COLAB = "google.colab" in sys.modules
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "llama3.1:8b"
STACK_URL = "http://127.0.0.1:8321"
STACK_MODEL = "llama3.1:8b"
STACK_VERSION = "0.7.3"
CLIENT_VERSION = "0.7.3"
RUNTIME_DIR = Path("/content/.veterinaria-phase1")

@dataclass(frozen=True)
class AppConfig:
    mode: str
    whatsapp_access_token: str | None = field(default=None, repr=False)
    whatsapp_phone_number_id: str | None = field(default=None, repr=False)
    whatsapp_verify_token: str | None = field(default=None, repr=False)
    ngrok_auth_token: str | None = field(default=None, repr=False)

    @property
    def transport_configured(self) -> bool:
        return all((self.whatsapp_access_token, self.whatsapp_phone_number_id,
                    self.whatsapp_verify_token, self.ngrok_auth_token))

def load_config(mode: str) -> AppConfig:
    if mode == LOCAL_TEST_MODE:
        return AppConfig(mode=mode)  # Sin importar userdata ni leer variables Meta.
    if mode != WHATSAPP_MODE:
        raise ValueError("Modo desconocido.")
    if not IN_COLAB:
        raise RuntimeError("La configuración WhatsApp usa Colab Secrets; ejecutar en Colab.")
    from google.colab import userdata
    # Nombres elegidos por esta implementación: el notebook no prescribe nombres.
    names = {
        "whatsapp_access_token": "WHATSAPP_ACCESS_TOKEN",
        "whatsapp_phone_number_id": "WHATSAPP_PHONE_NUMBER_ID",
        "whatsapp_verify_token": "WHATSAPP_VERIFY_TOKEN",
        "ngrok_auth_token": "NGROK_AUTH_TOKEN",
    }
    values, missing = {}, []
    for attribute, name in names.items():
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            value = None
        if not isinstance(value, str) or not value.strip():
            missing.append(name)
        else:
            values[attribute] = value.strip()
    if missing:
        raise RuntimeError("Configurar/autorizar estos Secrets: " + ", ".join(missing))
    return AppConfig(mode=mode, **values)

config = load_config(MODE)
print("CONFIG .............", config.mode)
# No mostrar config.__dict__, asdict(config) ni los valores de Secrets.

#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama — soporte de implementación, fase 1.
import shutil
import subprocess
import urllib.request

def require_colab():
    if not IN_COLAB or sys.platform != "linux":
        raise RuntimeError("Instalación preparada para Google Colab Linux; no ejecutar en Windows.")

def install_ollama():
    require_colab()
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    if shutil.which("ollama"):
        print("OLLAMA_INSTALL ..... ALREADY AVAILABLE")
        return
    # El instalador oficial de Linux utiliza archivos comprimidos zstd.
    missing = [name for name in ("curl", "zstd") if not shutil.which(name)]
    if missing:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *missing], check=True)
    installer = RUNTIME_DIR / "install-ollama.sh"
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    subprocess.run(["sh", str(installer)], check=True)
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama no quedó disponible en PATH.")
    print("OLLAMA_INSTALL ..... OK")

install_ollama()

In [ ]:
# Iniciar el servidor de Ollama en segundo plano y preparar Llama 3.1 8B.
import json
import socket
import time
from urllib.error import URLError

_processes = globals().get("_processes", {})  # Conservar al reejecutar.

def get_json(url, timeout=3):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.load(response)

def service_ready(url):
    try:
        get_json(url)
        return True
    except (URLError, TimeoutError, OSError, ValueError):
        return False

def port_in_use(port):
    with socket.socket() as connection:
        connection.settimeout(0.5)
        return connection.connect_ex(("127.0.0.1", port)) == 0

def service_environment():
    return {k: v for k, v in os.environ.items()
            if not k.startswith(("WHATSAPP_", "NGROK_"))}

def start_service(name, command, ready_url, port, *, env=None, timeout=120):
    require_colab()
    if service_ready(ready_url):
        return
    process = _processes.get(name)
    if process is None or process.poll() is not None:
        if port_in_use(port):
            raise RuntimeError(f"Puerto {port} ocupado por un servicio no listo; revisar.")
        RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
        with (RUNTIME_DIR / f"{name}.log").open("ab") as log:
            process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                       env=env, start_new_session=True)
        _processes[name] = process
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"{name} terminó; revisar su log en {RUNTIME_DIR}. No publicarlo sin revisar.")
        if service_ready(ready_url):
            return
        time.sleep(0.5)  # Polling acotado; no espera fija de arranque.
    raise TimeoutError(f"{name} no estuvo listo en {timeout}s; reejecutar reutiliza el proceso.")

def prepare_ollama_model():
    require_colab()
    environment = service_environment()
    environment["OLLAMA_HOST"] = "127.0.0.1:11434"
    start_service("ollama", ["ollama", "serve"], OLLAMA_URL + "/api/tags", 11434, env=environment)
    available = {m.get("name") for m in get_json(OLLAMA_URL + "/api/tags").get("models", [])}
    if OLLAMA_MODEL not in available:
        print("LLAMA_MODEL ........ DOWNLOADING (puede tardar varios minutos)")
        subprocess.run(["ollama", "pull", OLLAMA_MODEL], env=environment, check=True)
    available = {m.get("name") for m in get_json(OLLAMA_URL + "/api/tags").get("models", [])}
    if OLLAMA_MODEL not in available:
        raise RuntimeError("El modelo requerido no está disponible en Ollama.")
    print("OLLAMA ............. OK")
    print("LLAMA_MODEL ........ OK:", OLLAMA_MODEL)

prepare_ollama_model()

#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# Instalar dependencias — servidor aislado para no reemplazar paquetes de Colab.
from importlib import metadata

STACK_ENV = RUNTIME_DIR / "stack-env"
STACK_PYTHON = STACK_ENV / "bin/python"

def install_stack_dependencies():
    require_colab()
    if sys.version_info < (3, 12):
        raise RuntimeError("Llama Stack 0.7.3 requiere Python >=3.12. Usar un runtime Colab compatible.")
    if not STACK_PYTHON.exists():
        subprocess.run([sys.executable, "-m", "venv", str(STACK_ENV)], check=True)
    # Servidor/API/cliente alineados; solo dependencias de los providers utilizados.
    subprocess.run([str(STACK_PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                    f"llama-stack=={STACK_VERSION}", f"llama-stack-api=={STACK_VERSION}",
                    f"llama-stack-client=={CLIENT_VERSION}", "ollama==0.6.3",
                    "matplotlib", "pillow>=12.2.0", "scikit-learn", "mcp>=1.23.0,<2"], check=True)
    imported = sys.modules.get("llama_stack_client")
    if imported is not None and getattr(imported, "__version__", None) != CLIENT_VERSION:
        raise RuntimeError("Otra SDK ya importada: reiniciar el runtime antes de instalar el cliente.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    f"llama-stack-client=={CLIENT_VERSION}"], check=True)
    if metadata.version("llama-stack-client") != CLIENT_VERSION:
        raise RuntimeError("Versión inesperada del cliente.")
    print("STACK_PACKAGES ..... INSTALLED; readiness pendiente de la siguiente celda")

install_stack_dependencies()

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado.
# Dependencias instaladas en la celda anterior. Sin generar conversaciones.
import inspect

def build_stack_config():
    # Persistencia nativa de Stack: el agente NO consulta SQLite ni crea memoria
    # alternativa. Las conversaciones se administrarán exclusivamente por su API.
    return {
        "version": 2, "distro_name": "veterinaria-colab",
        "apis": ["inference", "responses", "vector_io", "tool_runtime", "files"],
        "providers": {
            "inference": [{"provider_id": "ollama", "provider_type": "remote::ollama",
                           "config": {"base_url": OLLAMA_URL + "/v1"}}],
            "responses": [{"provider_id": "builtin", "provider_type": "inline::builtin",
                           "config": {"persistence": {
                               "agent_state": {"backend": "kv_default", "namespace": "agents"},
                               "responses": {"backend": "sql_default", "table_name": "responses"}}}}],
            "files": [{"provider_id": "localfs", "provider_type": "inline::localfs",
                       "config": {"storage_dir": str(RUNTIME_DIR / "files"),
                                  "metadata_store": {"backend": "sql_default", "table_name": "files"}}}],
            # Dependencias internas de responses, sin RAG ni herramientas.
            "vector_io": [], "tool_runtime": [],
        },
        "storage": {"backends": {
            "kv_default": {"type": "kv_sqlite", "db_path": str(RUNTIME_DIR / "stack-kv.db")},
            "sql_default": {"type": "sql_sqlite", "db_path": str(RUNTIME_DIR / "stack-sql.db")},
        }, "stores": {
            "metadata": {"backend": "kv_default", "namespace": "registry"},
            "inference": {"backend": "sql_default", "table_name": "inference"},
            "conversations": {"backend": "sql_default", "table_name": "conversations"},
            "responses": {"backend": "sql_default", "table_name": "responses"},
            "prompts": {"backend": "kv_default", "namespace": "prompts"},
            "connectors": {"backend": "kv_default", "namespace": "connectors"},
        }},
        "registered_resources": {"models": [{"model_id": STACK_MODEL, "provider_id": "ollama",
                                             "provider_model_id": OLLAMA_MODEL, "model_type": "llm"}]},
        "server": {"host": "127.0.0.1", "port": 8321, "workers": 1},
    }

def prepare_stack_client():
    require_colab()
    if not service_ready(OLLAMA_URL + "/api/tags"):
        raise RuntimeError("Preparar Ollama y el modelo antes de Llama Stack.")
    stack_config = RUNTIME_DIR / "stack-config.json"
    serialized = json.dumps(build_stack_config(), indent=2)
    if stack_config.exists() and stack_config.read_text() != serialized and port_in_use(8321):
        raise RuntimeError("La configuración cambió con Stack activo. Reiniciar el servicio.")
    stack_config.write_text(serialized)
    # Esquema REAL instalado. JSON también es YAML válido para la CLI.
    validate = ("import json,sys; from llama_stack.core.datatypes import StackConfig; "
                "StackConfig.model_validate(json.load(open(sys.argv[1])))")
    subprocess.run([str(STACK_PYTHON), "-c", validate, str(stack_config)], check=True)
    start_service("llama-stack", [str(STACK_ENV / "bin/llama"), "stack", "run", str(stack_config)],
                  STACK_URL + "/v1/health", 8321, env=service_environment(), timeout=180)
    if get_json(STACK_URL + "/v1/version").get("version") != STACK_VERSION:
        raise RuntimeError("El servidor activo no coincide con la versión fijada.")
    if metadata.version("llama-stack-client") != CLIENT_VERSION:
        raise RuntimeError("Cliente incompatible con la versión fijada.")
    # Crear cliente SOLO después de readiness.
    from llama_stack_client import LlamaStackClient
    candidate = LlamaStackClient(base_url=STACK_URL, timeout=120.0, max_retries=0)
    try:
        required = [candidate.conversations.create, candidate.conversations.items.list,
                    candidate.conversations.items.delete, candidate.responses.create]
        if not all(callable(method) for method in required):
            raise RuntimeError("La SDK instalada no expone las APIs requeridas.")
        inspect.signature(candidate.conversations.create).bind()
        if "conversation" not in inspect.signature(candidate.responses.create).parameters:
            raise RuntimeError("responses.create no admite conversation.")
        paths = get_json(STACK_URL + "/openapi.json").get("paths", {})
        if any("post" not in paths.get(route, {}) for route in ("/v1/conversations", "/v1/responses")):
            raise RuntimeError("El servidor no expone conversations/responses.")
        if STACK_MODEL not in {m.id for m in candidate.models.list().data}:
            raise RuntimeError("Llama 3.1 8B no está registrado en Llama Stack.")
    except Exception:
        candidate.close()
        raise
    previous = globals().get("client")
    if previous is not None:
        previous.close()
    print("LLAMA_STACK ........ OK")
    print("CONVERSATIONS_API .. AVAILABLE (sin crear conversaciones)")
    return candidate

client = prepare_stack_client()

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# Configurar el servidor del webhook


In [ ]:
# Definir función central de normalización — compartida por ambos modos.
import re

def normalize_phone(phone_number: str) -> str:
    '''Clave internacional +dígitos; entrada sin '+' debe incluir código de país.

    Validación sintáctica, no certifica asignación ni existencia del número.
    No añade país ni aplica conversiones exclusivas de México.
    '''
    if not isinstance(phone_number, str):
        raise TypeError("El teléfono debe ser string.")
    value = phone_number.strip()
    if not value or not re.fullmatch(r"[+0-9().\s-]+", value):
        raise ValueError("Teléfono vacío o con caracteres no permitidos.")
    value = re.sub(r"[().\s-]", "", value)
    if value.startswith("00"):
        value = "+" + value[2:]
    digits = value[1:] if value.startswith("+") else value
    if not re.fullmatch(r"[1-9][0-9]{6,14}", digits):
        raise ValueError("Usar entre 7 y 15 dígitos con código internacional, sin extensiones.")
    return "+" + digits

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [ ]:
# Una conversacion de Llama Stack por numero de telefono


#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria


## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificacion de credenciales


In [ ]:
# Definir endpoint de recepcion de mensajes


In [ ]:
# Definir funcion de envio de respuesta


In [ ]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# Instalar e iniciar tunel publico con ngrok


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [ ]:
# Mandar el primer mensaje, mencionando el servicio


In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio


In [ ]:
# Verificar respuestas


In [ ]:
# Ver mensajes recibidos


In [ ]:
# Verificar almacenamiento en Llama Stock


In [ ]:
# Finalizar proceso de tunelización


## **CHALLENGE: AGENTE DE UNA VETERINARIA**

Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para una veterinaria: además de la regla de escalamiento a una persona, se agrega una regla de urgencia médica, y un límite al historial que se conserva en la conversación de Llama Stack.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)
# Fase 1: ejecutar primero el entorno Hands-On; reutilizar sus funciones.
install_ollama()

In [ ]:
# Iniciar el servidor de Ollama en segundo plano
# Reutilizar servicio/modelo; no iniciar segunda instancia.
prepare_ollama_model()

In [ ]:
# Instalar dependencias
# Mismo entorno aislado y versiones del Hands-On.
install_stack_dependencies()

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado
# Reutilizar servicio verificando readiness y APIs.
client = prepare_stack_client()

In [ ]:
# Leer credenciales desde Colab Secrets
# MODE se define en el Hands-On; modo local no consulta Secrets.
config = load_config(MODE)

**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [ ]:
# Conversacion de Llama Stack por numero de telefono


In [ ]:
# Definir la funcion de generacion de respuesta con memoria


**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario


In [ ]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama


**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [ ]:
# Configurar el servidor del webhook


In [ ]:
# Definir función de normalización de número
# La única implementación es normalize_phone, definida en Hands-On.
# Webhook y pruebas locales deberán llamar a esa misma función.
assert callable(normalize_phone)

In [ ]:
# Definir endpoint de verificacion de credenciales


In [ ]:
# Definir endpoint de recepcion de mensajes


In [ ]:
# Definir funcion de envio de respuesta


**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [ ]:
# Instalar e iniciar tunel publico con ngrok


In [ ]:
# Mandar el primer mensaje, mencionando el servicio


In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio


In [ ]:
# Verificar respuestas


In [ ]:
# Ver mensajes recibidos


In [ ]:
# Verificar almacenamiento en Llama Stack


In [ ]:
# Finalizar proceso de tunelización


## SOPORTE DE IMPLEMENTACIÓN — FASE 1

Estas celdas auxiliares complementan el ejercicio; las instrucciones originales permanecen intactas.
Ejecutar en orden el entorno Hands-On (6, 8, 9, 11, 12) y la normalización (15).
El Challenge reutiliza esas funciones; no es una segunda instalación independiente.
Se requiere Colab Linux con Python >=3.12, espacio para Llama 3.1 8B y recursos suficientes.
Una GPU es recomendable; no se garantiza velocidad en CPU. La descarga puede tardar varios minutos.
El runtime y el almacenamiento nativo de Stack son efímeros al eliminar la sesión Colab.

**Modos y Secrets:** LOCAL_TEST_MODE no consulta Meta ni arranca ngrok.
WHATSAPP_MODE exige WHATSAPP_ACCESS_TOKEN, WHATSAPP_PHONE_NUMBER_ID,
WHATSAPP_VERIFY_TOKEN y NGROK_AUTH_TOKEN en Colab Secrets, con acceso autorizado.
Son nombres elegidos por esta implementación. No pegar valores ni números personales en celdas/outputs.
Configurar Secrets no acredita la integración: webhook, BackgroundTasks, ngrok y envío quedan pendientes.

**Versiones:** servidor/API/cliente Llama Stack 0.7.3; proveedor Python Ollama 0.6.3.
Servidor en entorno virtual aislado; cliente en el kernel. Se limita MCP a >=1.23.0,<2: MCP 2 elimina McpError, importado por Stack 0.7.3.
Las demás dependencias transitivas no son un lockfile.
Se comprueban métodos instalados y rutas del servidor antes de declarar el entorno listo.
La ejecución completa sigue pendiente en Colab. La persistencia es la nativa de Llama Stack:
no se consulta directamente una base de datos ni se reemplaza conversations por memoria propia.

Referencias:
[paquete Llama Stack](https://pypi.org/project/llama-stack/0.7.3/),
[cliente](https://pypi.org/project/llama-stack-client/0.7.3/),
[Ollama Linux](https://docs.ollama.com/linux),
[modelo llama3.1:8b](https://ollama.com/library/llama3.1:8b).

**Arquitectura prevista:** celda local → process_message → texto/diagnóstico;
WhatsApp → webhook/tarea de fondo → la misma process_message → envío WhatsApp.
Normalización central; envío Meta fuera del núcleo. Conversaciones por teléfono,
reglas, memoria, locks y procesamiento aún NO implementados.

In [ ]:
# Diagnóstico de entorno: solo estado no sensible, nunca configuración completa.
def diagnose_environment():
    ollama_ok = service_ready(OLLAMA_URL + "/api/tags")
    model_ok = False
    stack_ok = False
    if ollama_ok:
        model_ok = OLLAMA_MODEL in {
            m.get("name") for m in get_json(OLLAMA_URL + "/api/tags").get("models", [])}
    if service_ready(STACK_URL + "/v1/health") and globals().get("client") is not None:
        try:
            stack_ok = (get_json(STACK_URL + "/v1/version").get("version") == STACK_VERSION
                        and STACK_MODEL in {m.id for m in client.models.list().data})
        except Exception:
            stack_ok = False
    ready = IN_COLAB and ollama_ok and model_ok and stack_ok
    rows = {
        "COLAB": "OK" if IN_COLAB else "NOT RUNNING IN COLAB",
        "OLLAMA": "OK" if ollama_ok else "NOT READY",
        "LLAMA_MODEL": "OK" if model_ok else "NOT READY",
        "LLAMA_STACK": "OK" if stack_ok else "NOT READY",
        "LOCAL_TEST_MODE": "ENVIRONMENT READY" if ready else "ENVIRONMENT NOT READY",
        "WHATSAPP_MODE": "SECRETS CONFIGURED; INTEGRATION PENDING" if config.transport_configured else "NOT CONFIGURED",
        "PROCESS_MESSAGE": "NOT IMPLEMENTED (fase siguiente)",
    }
    for name, status in rows.items():
        print(f"{name:.<24} {status}")

diagnose_environment()

In [ ]:
# Contratos e instrumentación — soporte fase 1, sin lógica conversacional.
from collections import deque
from dataclasses import dataclass, field
from enum import Enum
from uuid import uuid4
import time

class Route(str, Enum):
    LLM = "LLM"
    HUMAN_ESCALATION = "HUMAN_ESCALATION"
    VETERINARY_EMERGENCY = "VETERINARY_EMERGENCY"

@dataclass(frozen=True)
class CoreResult:
    response_text: str
    normalized_phone: str = field(repr=False)
    conversation_id: str | None = field(repr=False)
    route: Route
    history_count: int | None  # None = no observado, no un cero inventado.
    run_id: str

class EventKind(str, Enum):
    RUN_STARTED = "RUN_STARTED"
    LOCK_WAITING = "LOCK_WAITING"
    LOCK_ACQUIRED = "LOCK_ACQUIRED"
    LOCK_RELEASED = "LOCK_RELEASED"
    GENERATION_REQUESTED = "GENERATION_REQUESTED"
    GENERATION_COMPLETED = "GENERATION_COMPLETED"
    HISTORY_OBSERVED = "HISTORY_OBSERVED"
    ROUTE_SELECTED = "ROUTE_SELECTED"
    RUN_FINISHED = "RUN_FINISHED"

@dataclass(frozen=True)
class DiagnosticEvent:
    event: EventKind
    run_id: str
    timestamp_ns: int
    normalized_phone: str | None = field(default=None, repr=False)
    conversation_id: str | None = field(default=None, repr=False)
    route: Route | None = None
    history_count: int | None = None

class DiagnosticRecorder:
    # Registro acotado y opt-in, NO memoria conversacional.
    # En pruebas usar solo teléfonos sintéticos. No registrar texto/payloads/Secrets.
    # Emitir eventos de generación junto a llamadas REALES en la siguiente fase.
    def __init__(self, *, enabled=False, capacity=1000):
        self.enabled = enabled
        self._events = deque(maxlen=capacity)

    def record(self, event: EventKind, *, run_id: str, normalized_phone=None,
               conversation_id=None, route=None, history_count=None):
        if not self.enabled:
            return
        if not isinstance(event, EventKind):
            raise TypeError("Evento no permitido.")
        if route is not None and not isinstance(route, Route):
            raise TypeError("Ruta no permitida.")
        if history_count is not None and (type(history_count) is not int or history_count < 0):
            raise ValueError("history_count debe ser entero no negativo o None.")
        self._events.append(DiagnosticEvent(event, run_id, time.monotonic_ns(),
                                            normalized_phone, conversation_id, route, history_count))

    def snapshot(self):
        return tuple(self._events)

    def clear(self):
        self._events.clear()

def new_run_id() -> str:
    return uuid4().hex

diagnostics = DiagnosticRecorder(enabled=False)

def process_message(phone_number: str, message: str) -> CoreResult:
    # Contrato único para Colab/webhook. Envío a Meta siempre fuera del núcleo.
    # Pendiente: normalizar, locks, conversaciones reales, reglas y generación.
    raise NotImplementedError("process_message pendiente de la siguiente fase; sin respuesta simulada.")

## PRUEBAS LOCALES — SIN META

Soporte de implementación. Ya pueden probarse configuración local, contratos y normalización.
El procesamiento queda reservado: invocarlo ahora produce NotImplementedError.
No hay resultados exitosos prefabricados.

Pruebas futuras pendientes, con Ollama, modelo, Llama Stack, conversations, locks y reglas reales:

1. **Memoria del mismo teléfono:** continuidad contextual y conversación reutilizada.
2. **Aislamiento entre teléfonos:** IDs e historiales independientes.
3. **Escalamiento humano:** ruta HUMAN_ESCALATION y mensaje fijo.
4. **Urgencia veterinaria:** ruta VETERINARY_EMERGENCY y ninguna solicitud real de generación.
5. **Recorte real del historial:** superar el límite por definir e inspeccionar almacenamiento en Stack.
6. **Concurrencia por teléfono:** eventos de locks para el mismo teléfono y teléfonos distintos.

Solo se sustituirá el transporte WhatsApp. Estas pruebas no acreditan webhook/ngrok/Meta.

In [ ]:
# Campos editables: números sintéticos; nunca usarlos para envíos reales.
TEST_PHONE = "+15550000001"
TEST_MESSAGE = "Hola"

# Pendiente de la siguiente fase. Descomentar ahora produce NotImplementedError.
# result = process_message(TEST_PHONE, TEST_MESSAGE)
# print(result.response_text)  # Texto visible separado de metadata.

In [ ]:
# Pruebas de normalización sin Colab, Meta, Ollama ni Llama Stack.
assert normalize_phone(TEST_PHONE) == "+15550000001"
assert normalize_phone(" +1 (555) 000-0001 ") == normalize_phone("15550000001")
assert normalize_phone("0015550000001") == "+15550000001"
assert normalize_phone("+44 7700 900123") == "+447700900123"  # Número de ficción británico.
assert normalize_phone(normalize_phone(TEST_PHONE)) == normalize_phone(TEST_PHONE)
for invalid in ("", "   ", "+", "123", "+0123456789", "++15550000001", "abc", "+15550000001 ext 2", "1" * 16):
    try:
        normalize_phone(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Se aceptó una entrada inválida.")
try:
    normalize_phone(15550000001)
except TypeError:
    pass
else:
    raise AssertionError("Debe exigir string.")
print("NORMALIZATION ...... OK")